<a href="https://colab.research.google.com/github/kachionyenokwe/TempSound/blob/main/notebooks/01_yamnet_sed_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install yt-dlp for audio fetching and TensorFlow Hub for YAMNet
!pip install -q yt-dlp tensorflow-hub librosa soundfile

import os
import json
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
import librosa
import soundfile as sf
import yt_dlp

print("✓ Environment dependencies loaded successfully.")
print("TensorFlow Version:", tf.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.6 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


✓ Environment dependencies loaded successfully.
TensorFlow Version: 2.21.0


In [4]:
# Load pre-trained YAMNet model from TensorFlow Hub
print("[*] Loading YAMNet model from TFHub...")
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')

# Load YAMNet class mapping (521 AudioSet event classes)
class_map_path = yamnet_model.class_map_path().numpy().decode('utf-8')
class_names = []

# Use standard tf.io.gfile.GFile (capital 'F') or standard open
with tf.io.gfile.GFile(class_map_path) as csv_file:
    import csv
    reader = csv.DictReader(csv_file)
    for row in reader:
        class_names.append(row['display_name'])

class_names = np.array(class_names)
print(f"[✓] YAMNet loaded successfully with {len(class_names)} audio event classes.")

[*] Loading YAMNet model from TFHub...
[✓] YAMNet loaded successfully with 521 audio event classes.


In [5]:
# Create working directories
os.makedirs("audio_downloads", exist_ok=True)
os.makedirs("output", exist_ok=True)

# Option A: Upload manually if running standalone in Colab
from google.colab import files
if not os.path.exists("gold_standard_50_sed_timestamps.json"):
    print("[*] Please upload 'gold_standard_50_sed_timestamps.json':")
    uploaded = files.upload()

with open("gold_standard_50_sed_timestamps.json", "r", encoding="utf-8") as f:
    manifest = json.load(f)

print(f"[✓] Loaded manifest with {len(manifest)} Gold Standard prompts.")

[*] Please upload 'gold_standard_50_sed_timestamps.json':


Saving gold_standard_50_sed_timestamps.json to gold_standard_50_sed_timestamps.json
[✓] Loaded manifest with 50 Gold Standard prompts.


In [6]:
def download_audio_clip(youtube_id, output_dir="audio_downloads"):
    """
    Downloads audio waveform for a given YouTube ID and converts to 16kHz mono WAV.
    """
    output_path = os.path.join(output_dir, f"{youtube_id}.wav")
    if os.path.exists(output_path):
        return output_path

    url = f"https://www.youtube.com/watch?v={youtube_id}"
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'outtmpl': os.path.join(output_dir, f"{youtube_id}.%(ext)s"),
        'quiet': True,
        'no_warnings': True
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])

        # Resample to 16kHz mono as required by YAMNet
        if os.path.exists(output_path):
            audio, sr = librosa.load(output_path, sr=16000, mono=True)
            sf.write(output_path, audio, 16000)
            return output_path
    except Exception as e:
        print(f"[Warning] Failed to download {youtube_id}: {e}")
        return None

In [7]:
CONFIDENCE_THRESHOLD = 0.20  # Minimum confidence score for detected event

def process_audio_with_yamnet(wav_path):
    """
    Executes YAMNet frame inference and extracts event onset/offset timestamps.
    YAMNet uses 0.96s frames sliding every 0.48s.
    """
    audio_data, sr = librosa.load(wav_path, sr=16000, mono=True)

    # YAMNet inference
    scores, embeddings, spectrogram = yamnet_model(audio_data)
    scores = scores.numpy()  # Shape: (N_frames, 521)

    detected_events = []
    num_frames = scores.shape[0]

    for frame_idx in range(num_frames):
        # YAMNet frame timing: 0.48s step size
        t_start = round(frame_idx * 0.48, 2)
        t_end = round(t_start + 0.96, 2)

        top_class_idx = np.argmax(scores[frame_idx])
        confidence = float(scores[frame_idx][top_class_idx])

        if confidence >= CONFIDENCE_THRESHOLD:
            label = class_names[top_class_idx]

            # Avoid duplicate consecutive frame logging for the same class
            if detected_events and detected_events[-1]["label"] == label:
                detected_events[-1]["t_end"] = t_end
            else:
                detected_events.append({
                    "label": label,
                    "confidence": round(confidence, 3),
                    "t_start": t_start,
                    "t_end": t_end
                })

    return detected_events

In [8]:
print("[*] Starting batch SED timestamp extraction...")

for entry in manifest:
    yt_id = entry.get("youtube_id")
    sample_id = entry.get("sample_id")
    caption = entry.get("caption")

    print(f"\nProcessing Sample {sample_id}/50: '{caption}'")

    # 1. Fetch Audio
    wav_file = download_audio_clip(yt_id)

    if wav_file and os.path.exists(wav_file):
        # 2. Extract SED Timestamps
        events = process_audio_with_yamnet(wav_file)
        entry["detected_events"] = events
        print(f" -> Extracted {len(events)} event intervals.")
    else:
        print(f" -> [Skipped] Audio unavailable for YouTube ID: {yt_id}")

# Export populated ground-truth manifest
output_manifest_path = "output/gold_standard_50_sed_timestamps_populated.json"
with open(output_manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

print(f"\n[✓] SED Extraction Complete!")
print(f"[✓] Populated manifest saved to: {output_manifest_path}")

# Trigger automatic download back to host machine
files.download(output_manifest_path)

[*] Starting batch SED timestamp extraction...

Processing Sample 1/50: 'a man speaks with slight accent as a crowd in the background makes noise then silences'


ERROR: unable to download video data: HTTP Error 403: Forbidden


[Warning] Failed to download 0Jw-3txze7g: ERROR: unable to download video data: HTTP Error 403: Forbidden
 -> [Skipped] Audio unavailable for YouTube ID: 0Jw-3txze7g

Processing Sample 2/50: 'a vehicle accelerates and a man yells then a siren sounds'
 -> Extracted 55 event intervals.

Processing Sample 3/50: 'a vehicle engine revs and accelerates whining then a man laughs'


ERROR: [youtube] cEYr8peuyaQ: This video is unavailable


[Warning] Failed to download cEYr8peuyaQ: ERROR: [youtube] cEYr8peuyaQ: This video is unavailable
 -> [Skipped] Audio unavailable for YouTube ID: cEYr8peuyaQ

Processing Sample 4/50: 'a man is speaking and then a gun is fired and cocked'
 -> Extracted 143 event intervals.

Processing Sample 5/50: 'many loud squeaks or whistles sounding together, a vehicle passing, then a vehicle horn accompanies a male voice'
 -> Extracted 257 event intervals.

Processing Sample 6/50: 'a man speaks while a water tap runs then a beep occurs'
 -> Extracted 167 event intervals.

Processing Sample 7/50: 'a motorcycle engine idles then turns off'
 -> Extracted 148 event intervals.

Processing Sample 8/50: 'a woman speaks and then a kid yells loudly as soft voices speak softly in the background'
 -> Extracted 129 event intervals.

Processing Sample 9/50: 'an engine idles as a man speaks then it accelerates'
 -> Extracted 6 event intervals.

Processing Sample 10/50: 'a motorcycle engine is running then someon

...

[Warning] Failed to download iFPgF3A3omo: ERROR: unable to download video data: HTTP Error 403: Forbidden
 -> [Skipped] Audio unavailable for YouTube ID: iFPgF3A3omo

Processing Sample 37/50: 'a thunder storm is present while traffic honking is present'
 -> Extracted 41 event intervals.

Processing Sample 38/50: 'horses walk causing clip-clopping while a man speaks'
 -> Extracted 58 event intervals.

Processing Sample 39/50: 'in a quiet environment, slow ticking is ongoing while an adult male speaks'
 -> Extracted 2 event intervals.

Processing Sample 40/50: 'a dog is barking while a man and woman make indistinguishable words over the barking'
 -> Extracted 33 event intervals.

Processing Sample 41/50: 'a man speaking with brief tapping in the background followed by high pitched beeping'
 -> Extracted 233 event intervals.

Processing Sample 42/50: 'several people talk followed by a gun fire'
 -> Extracted 48 event intervals.

Processing Sample 43/50: 'wind blowing into a microphone wit

ERROR: [youtube] GPN4HyzFShY: Private video


[Warning] Failed to download GPN4HyzFShY: ERROR: [youtube] GPN4HyzFShY: Private video
 -> [Skipped] Audio unavailable for YouTube ID: GPN4HyzFShY

Processing Sample 47/50: 'an engine vibrates followed by a chainsaw revving'
 -> Extracted 5 event intervals.

Processing Sample 48/50: 'water is flowing forcefully followed by a toilet loudly flushing'
 -> Extracted 30 event intervals.

Processing Sample 49/50: 'an organ briefly playing followed by a whooshing sound effect'
 -> Extracted 17 event intervals.

Processing Sample 50/50: 'a male communicating with another male outside, followed by insects flying and buzzing in the background'
 -> Extracted 39 event intervals.

[✓] SED Extraction Complete!
[✓] Populated manifest saved to: output/gold_standard_50_sed_timestamps_populated.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>